In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
from plotly import graph_objects as go

import nn_gmm as nng

In [ ]:
sel_ruptures_ffp = Path("/Users/claudy/dev/work/data/nn_gmm/ds_simulations/disagg/selected_ruptures.csv")

In [ ]:
sel_ruptures_df = pd.read_csv(sel_ruptures_ffp, index_col=0)
im_dir = Path("/Users/claudy/dev/work/data/nn_gmm/ds_simulations/ims")
completed_events = np.array([int(item.name) for item in im_dir.iterdir() if item.is_dir()])

In [ ]:
df = sel_ruptures_df.copy()
df["completed"] = df.index.isin(completed_events)


## Magnitude Distribution

In [ ]:
unique_mag_values = np.unique(df.mag)
mag_bins = np.linspace(start=unique_mag_values.min() - 0.05, stop=unique_mag_values.max() + 0.05, num=unique_mag_values.size + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

sns.histplot(data=df, x="mag", hue="completed", multiple="stack", bins=mag_bins)
ax.set_xlim(mag_bins.min(), mag_bins.max())

fig.tight_layout()

## Depth Distribution

In [ ]:
unique_depth_values = np.unique(df.depth)
depth_bins = np.linspace(start=unique_depth_values.min() - 10, stop=unique_depth_values.max() + 10, num=unique_depth_values.size + 1)

fig, ax = plt.subplots(figsize=(16, 8))

sns.histplot(data=df, x="depth", hue="completed", multiple="stack", bins=depth_bins)
ax.set_xlim(depth_bins.min(), depth_bins.max())

fig.tight_layout()

## Magnitude-Depth Distribution

In [ ]:
depth_bins_centers = (depth_bins[:-1] + depth_bins[1:]) / 2 
mag_bin_centres = (mag_bins[:-1] + mag_bins[1:]) / 2

In [ ]:
def create_mag_depth_disagg_plot(rp: int, ims: list[str], binned_disagg: xr.DataArray):
    for im in ims:
        cur_bin_disagg = binned_disagg.sel(im=im, rp=rp)
        cur_bin_disagg = cur_bin_disagg.mean(dim="site").to_pandas()

        create_single_mag_depth_disagg_plot(im, cur_bin_disagg)

def create_single_mag_depth_disagg_plot(im: str, cur_bin_disagg: pd.DataFrame, vmin: float =0, vmax: float =15, count_label: str = "Contribution (%)", multiplier: float =100):
    fig, axs = plt.subplot_mosaic(
        [[".", "histx", "."], ["cbar", "main", "histy"]],
        width_ratios=(0.1, 4, 1),
        height_ratios=(1, 4),
        layout="constrained",
        figsize=(16,8),
    )
    ax = axs["main"]
    ax_histx = axs["histx"]
    ax_histy = axs["histy"]
    ax_cbar = axs["cbar"]   

    r = ax.pcolormesh(mag_bin_centres, depth_bins_centers, cur_bin_disagg.values * multiplier, cmap="Blues", vmin=vmin, vmax=vmax)
    ax.set_xlabel("Magnitude")
    ax.set_ylabel("Depth (km)")
    ax.invert_yaxis()


    ax_histx.bar(mag_bin_centres, cur_bin_disagg.sum(axis=0).values * multiplier, width=np.diff(mag_bins), align="center", edgecolor="black")
    ax_histx.set_ylabel(count_label)
    ax_histx.set_xticklabels([])
    ax_histx.set_xlim(ax.get_xlim())

    ax_histy.barh(depth_bins_centers, cur_bin_disagg.sum(axis=1).values * multiplier, height=np.diff(depth_bins), align="center", edgecolor="black")
    ax_histy.set_xlabel(count_label)
    ax_histy.set_yticklabels([])
    ax_histy.set_ylim(ax.get_ylim())

    plt.colorbar(r, cax=ax_cbar, label=count_label, location="left")
    fig.text(0.97, 0.9, nng.utils.get_nice_im_name(im), ha="right", va="center", fontsize=16, fontweight="bold")

In [ ]:
sel_ruptures_dist = df.groupby(["depth_bin", "mag_bin"], observed=False, as_index=False).size().pivot_table(index="depth_bin", columns="mag_bin", values="size", aggfunc="sum", fill_value=0, observed=False)

### Selected Ruptures (1000)
The set of 1000 DS ruptures initially selected based on the disagg results.

In [ ]:
create_single_mag_depth_disagg_plot("Selected Ruptures", sel_ruptures_dist, vmin=0, vmax=30, count_label="Number of Ruptures", multiplier=1)

### Simulated Ruptures

In [ ]:
df = df.astype({"depth_bin": "category", "mag_bin": "category"})

In [ ]:
completed_ruptures_dist = df[df.completed].groupby(["depth_bin", "mag_bin"], observed=False, as_index=False).size().pivot_table(index="depth_bin", columns="mag_bin", values="size", aggfunc="sum", fill_value=0, observed=False)

In [ ]:
create_single_mag_depth_disagg_plot("Completed", completed_ruptures_dist, vmin=0, vmax=30, count_label="Number of Ruptures", multiplier=1)

## Spatial Distribution

In [ ]:
fig = go.Figure()
fig.update_layout(width=1600, height=800,
                  margin=dict(l=10, r=10, t=30, b=10),
                  map=dict(zoom=6, center=dict(lat=df.lat.mean(), lon=df.lon.mean())),
)

fig.add_trace(
    go.Scattermap(
        lon=df["lon"],
        lat=df["lat"],
        mode="markers",
        marker=dict(color="blue", size=4),
        name="Selected Ruptures",
    )
)

m = df.completed
fig.add_trace(
    go.Scattermap(
        lon=df.loc[m, "lon"],
        lat=df.loc[m, "lat"],
        mode="markers",
        marker=dict(color="red", size=4),
        name="Simulated Ruptures",
    )
)

fig.show()